<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day05-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 5 — In-class discussion problem (1 of 3)

Work this out **by hand in your group first** — then run the code cell to check your answer before presenting.

## Why does GKV have so many fewer neighbor words than LWG?

The book page found 49 words scoring &ge;12 against the query word `LWG` using real BLOSUM62 scores. The word `GKV` -- which occurs in *HBB*'s own sequence, right next to `LWG` (`...TALWGKVNVDEV...`) -- is a different query word of the same length.

1. Look up BLOSUM62's diagonal (self-substitution) scores for L, W, G, K, and V. Which single residue's diagonal score stands out as unusually high compared to the rest?
2. `LWG`'s and `GKV`'s own self-scores (word vs. itself) are the sum of three diagonal values each. Compute both sums by hand.
3. Predict: will `GKV` have *more* or *fewer* neighbor words above a threshold of T=11 than `LWG` had at T=12, and why might that follow directly from question 1?
4. Run the cell below to check.

In [1]:
from Bio.Align import substitution_matrices
from itertools import product

blosum62 = substitution_matrices.load("BLOSUM62")
AA = sorted(set(blosum62.alphabet) - {"*", "X", "B", "Z"})

def word_score(w1, w2):
    return sum(blosum62[a, b] for a, b in zip(w1, w2))

print("Diagonal (self-substitution) scores:")
for r in "LWGKV":
    print(f"  {r}-{r}: {blosum62[r, r]}")

for query_word, T in [("LWG", 12), ("GKV", 11)]:
    scores = {"".join(w): word_score(query_word, "".join(w)) for w in product(AA, repeat=3)}
    above = [w for w, s in scores.items() if s >= T]
    print(f"\n{query_word}: {len(above)} words score >= {T}")

Diagonal (self-substitution) scores:
  L-L: 4.0
  W-W: 11.0
  G-G: 6.0
  K-K: 5.0
  V-V: 4.0

LWG: 49 words score >= 12

GKV: 10 words score >= 11


**Discussion point:** tryptophan's BLOSUM62 diagonal score is +11 -- far above every other residue (the next-highest, cysteine, is +9; most residues are +4 to +6). Tryptophan is bulky and rare, so an exact match at a W position is much more "surprising", and therefore much more informative, than a match at a common residue. `LWG`'s self-score is 4+11+6=21, inflated almost entirely by that one W; `GKV`'s is only 6+5+4=15, with no similarly distinctive residue anywhere in it. A word built around a high-diagonal residue like W starts from a higher ceiling and therefore clears a fixed threshold with far more substituted variants than a word without one -- not because G, K, or V are individually hard to substitute, but because none of them gives the word the same head start W does.